# MiniDeepSeek — DeepSeek-style small char LM

Не полный DeepSeek-V3 MoE (сотни B). Для KizaruAi (~15–50M, один GPU) берём **один** паттерн из open DeepSeek papers:

1. **MLA** (Multi-Head Latent Attention) — Q и KV через low-rank latent, потом expand в heads.
2. **RMSNorm + SwiGLU** — pre-norm блок как у DeepSeek/Llama-подобных.
3. **Без MoE** — dense FFN; MoE-lite не тянем на char-корпусе этого размера.

Интерфейс как у `MiniGPT` / `SimpleTransformer`: `forward([B,T]) -> [B,T,V]`. Обучение: `python -m training.train_lm --arch deepseek ...`.

Код: `core/text/deepseek.py`. Чекпоинт: `checkpoints/deepseek.pt` (после следующего retrain).


## Почему MLA, а не MoE

| Идея DeepSeek | Здесь |
|---------------|--------|
| Full MoE / DeepSeekMoE | нет — эксперты раздувают params и требуют большой data/batch |
| MLA | да — меньше KV cache / params на attention при том же `d_model` |
| RMSNorm + SwiGLU | да — стандартный DeepSeek/Llama block |
| RoPE decoupled ranks | нет — absolute pos как у transformer/gpt в этом репо (совместимо с `train_lm`) |

Sketch MLA:

$$c_q = W^{DQ} x,\quad q = W^{UQ} c_q$$

$$c_{kv} = W^{DKV} x,\quad [k; v] = W^{UKV} c_{kv}$$

$$\mathrm{Attn} = \mathrm{softmax}(q k^\top / \sqrt{d} + M)\, v$$

где `M` — causal mask; `rank(c_kv) << d_model` (дефолт `d_model // 4`).


In [ ]:
from pathlib import Path

import torch

ROOT = Path.cwd().resolve()
if not (ROOT / "core").is_dir() and (ROOT.parent / "core").is_dir():
    ROOT = ROOT.parent

from core.text.deepseek import MiniDeepSeek

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
vocab = 64
model = MiniDeepSeek(
    vocab_size=vocab,
    d_model=64,
    n_layers=2,
    n_heads=4,
    block_size=32,
).to(device)
n_params = sum(p.numel() for p in model.parameters())
tokens = torch.randint(0, vocab, (2, 16), device=device)
logits = model(tokens)
print("device", device)
print("params", n_params)
print("logits", tuple(logits.shape))
print("kv_lora", model.kv_lora_rank, "q_lora", model.q_lora_rank)
assert logits.shape == (2, 16, vocab)


## Causal smoke

Позиции до изменённого токена должны совпасть (маска не даёт смотреть вперёд).


In [ ]:
model.eval()
left = torch.randint(0, vocab, (1, 12), device=device)
right = left.clone()
right[0, 7] = (right[0, 7] + 1) % vocab
with torch.no_grad():
    a = model(left)
    b = model(right)
ok_prefix = torch.allclose(a[0, :7], b[0, :7], atol=1e-5)
changed_tail = not torch.allclose(a[0, 7:], b[0, 7:], atol=1e-5)
print("causal_prefix_ok", bool(ok_prefix), "tail_differs", bool(changed_tail))
assert ok_prefix and changed_tail


## Retrain (когда корпус готов)

Не запускать, пока Genius/shared корпус не готов. Ориентир ёмкости как у transformer/gpt retrain:

```bash
python -m training.train_lm --arch deepseek --level char \
  --d-model 384 --n-layers 8 --n-heads 8 --block-size 384 \
  --steps 45000 --out checkpoints/deepseek.pt
```

Воркер после чекпоинта:

```bash
python -m app.workers.deepseek checkpoints/deepseek.pt --db data/bot.db
```

Discord: `/ai text model:deepseek`.
